# 01 - Download source PDFs and extract page text

**Before running**
1. Right panel -> *Session options* -> **Internet: On**. Accelerator can be **None** (CPU is enough).
2. *Add Input* -> upload a Kaggle Dataset containing `sources_manifest.json`
   (from `data/processed/` of the project).
3. To **resume** after a stopped/timed-out run: attach the previous version's output as an extra input;
   the notebook copies its `pdf_text/` folder and skips finished documents.

**What it does**: for each document in the manifest (most-cited first) it downloads the PDF, extracts per-page text with
PyMuPDF, scores extraction quality (flags scanned / legacy-font garbled / Indic-Unicode pages), writes compressed text shards to
`/kaggle/working/pdf_text/`, and **deletes the PDF** (Kaggle's working dir is capped at 20 GB).

In [7]:
!pip install -q pymupdf

In [10]:
%%writefile pdf_pipeline.py
"""Download source PDFs, extract per-page text, score extraction quality.

Designed for Kaggle (20 GB working-dir cap): PDFs are downloaded to a temp dir, text is extracted,
and the PDF is deleted unless keep_pdfs=True. Only compressed text shards are kept.

Outputs (out_dir):
  pages_XXXXX.jsonl.gz   one JSON per page: doc_id, page (1-based), n_chars, script, quality, sw, text
  status.jsonl           one JSON per processed doc (append-only; used for resume)
  doc_summary.csv        written by summarize(): one row per doc with quality verdict
"""
import csv
import gzip
import json
import os
import random
import re
import shutil
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import Pool
from urllib.parse import unquote

import requests

ZOHO_CDN = "https://files-accl.zohoexternal.in/public/workdrive-external/download/{}?x-cli-msg=null"
UA = {"User-Agent": "Mozilla/5.0 (compatible; academic-nlp-assignment)"}
MAX_BYTES = 400 * 1024 * 1024
DOC_DEADLINE_S = 240
EXTRACT_DOC_S = 120  # hard wall-clock limit per PDF; a hung extraction is killed and the doc marked extract_timeout
TERMINAL = {"ok", "not_pdf", "http_404", "http_403", "too_large", "extract_error", "extract_timeout"}

SCRIPTS = {
    "devanagari": (0x0900, 0x097F), "bengali": (0x0980, 0x09FF), "gurmukhi": (0x0A00, 0x0A7F),
    "gujarati": (0x0A80, 0x0AFF), "odia": (0x0B00, 0x0B7F), "tamil": (0x0B80, 0x0BFF),
    "telugu": (0x0C00, 0x0C7F), "kannada": (0x0C80, 0x0CFF), "malayalam": (0x0D00, 0x0D7F),
}
STOP = set("""the of and in to a is for with on are be by as at or from that this it an can should per
which also not may more than will have has been used use when if into these their such other all
during after before between each only one two high low early late crop plant plants""".split())
WORD_RE = re.compile(r"[A-Za-z]+")


# ----------------------------------------------------------------------------- download
def _safe_name(cd):
    if not cd:
        return ""
    m = re.search(r"filename\*=UTF-8''([^;]+)", cd) or re.search(r'filename="?([^";]+)"?', cd)
    return unquote(m.group(1)) if m else ""


def download(doc, dest_dir, session, retries=4):
    """Returns (status, path, info). status in {ok, not_pdf, http_NNN, timeout, too_large}."""
    url = ZOHO_CDN.format(doc["doc_id"]) if doc["kind"] == "zoho" else doc["url"]
    path = os.path.join(dest_dir, doc["doc_id"] + ".pdf")
    last = "timeout"
    deadline = time.time() + DOC_DEADLINE_S  # hard wall-clock cap across all attempts
    for attempt in range(retries):
        if time.time() > deadline:
            break
        try:
            with session.get(url, headers=UA, stream=True, timeout=(10, 30), allow_redirects=True) as r:
                if r.status_code != 200:
                    last = "http_%d" % r.status_code
                    if r.status_code in (403, 404):
                        return last, None, {}
                    raise RuntimeError(last)
                size = 0
                with open(path, "wb") as f:
                    for chunk in r.iter_content(1 << 16):
                        size += len(chunk)
                        if size > MAX_BYTES:
                            return "too_large", None, {}
                        if time.time() > deadline:  # trickling connection
                            raise TimeoutError("deadline")
                        f.write(chunk)
                info = {"bytes": size, "filename": _safe_name(r.headers.get("Content-Disposition")),
                        "ctype": r.headers.get("Content-Type", "")}
            with open(path, "rb") as f:
                head = f.read(1024)
            if b"%PDF-" not in head:
                os.remove(path)
                return "not_pdf", None, info
            return "ok", path, info
        except Exception as e:  # network error, timeout, 5xx/429
            last = last if last.startswith("http_") else "timeout"
            time.sleep(min(30, 2 ** attempt) + random.random())
    return last, None, {}


# ----------------------------------------------------------------------------- extraction
def page_stats(text):
    t = text.strip()
    n = len(t)
    if n < 40:
        return {"n_chars": n, "script": "none", "quality": "empty", "sw": 0.0}
    letters = [c for c in t if c.isalpha()]
    counts = Counter()
    for c in letters:
        o = ord(c)
        for name, (lo, hi) in SCRIPTS.items():
            if lo <= o <= hi:
                counts[name] += 1
                break
        else:
            counts["latin" if o < 0x250 else "other"] += 1
    nl = max(1, len(letters))
    top, topn = counts.most_common(1)[0] if counts else ("none", 0)
    # legacy-font garbage shows up as Latin-1 symbols / private-use / replacement chars
    weird = sum(1 for c in t if (0x80 <= ord(c) <= 0xBF) or (0xF000 <= ord(c) <= 0xF8FF) or c == " "
                or (0xC0 <= ord(c) <= 0xFF and top == "latin"))
    weird_r = weird / n
    toks = [w.lower() for w in WORD_RE.findall(t)]
    sw = sum(1 for w in toks if w in STOP) / max(1, len(toks))
    nonspace = max(1, sum(1 for c in t if not c.isspace()))
    digit_r = sum(c.isdigit() for c in t) / nonspace
    if top not in ("latin", "other") and topn / nl >= 0.4:
        q = "indic_unicode"
    elif weird_r > 0.05:
        q = "garbled"
    elif sw >= 0.12 and len(toks) >= 15:
        q = "ok"
    elif digit_r > 0.35:
        q = "numeric"
    elif len(toks) < 15:
        q = "sparse"
    else:
        q = "garbled"
    return {"n_chars": n, "script": top, "quality": q, "sw": round(sw, 3)}


def extract(path):
    """Worker: returns (doc_id, pages|None, err)."""
    import pymupdf as fitz  # `import fitz` is the deprecated alias and warns in every worker
    doc_id = os.path.basename(path)[:-4]
    try:
        d = fitz.open(path)
        if d.needs_pass and not d.authenticate(""):
            return doc_id, None, "encrypted"
        pages = []
        for i in range(d.page_count):
            try:
                text = d[i].get_text("text")
            except Exception:
                text = ""
            st = page_stats(text)
            st.update(doc_id=doc_id, page=i + 1, text=text)
            pages.append(st)
        d.close()
        return doc_id, pages, None
    except Exception as e:
        return doc_id, None, "%s: %s" % (type(e).__name__, str(e)[:120])


def _extract_worker(path, out_path):
    """Child process: extract one PDF and write the result to disk (so the parent can kill us safely)."""
    doc_id, pages, err = extract(path)
    with gzip.open(out_path, "wt", encoding="utf-8") as f:
        json.dump([pages, err], f, ensure_ascii=False)


def extract_all(paths, procs, per_doc_s=None):
    """Extract many PDFs, at most `procs` at a time, each in its own process with a hard wall-clock limit.
    A PDF that hangs (corrupt file, pathological page) is killed after per_doc_s and reported as 'timeout'
    without affecting the others. Returns {doc_id: (pages|None, err|None)}."""
    import multiprocessing as mp
    per_doc_s = per_doc_s or EXTRACT_DOC_S
    ctx = mp.get_context("spawn" if os.name == "nt" else "fork")
    results, pending, running = {}, list(paths), {}
    while pending or running:
        while pending and len(running) < procs:
            p = pending.pop(0)
            out = p[:-4] + ".json.gz"
            pr = ctx.Process(target=_extract_worker, args=(p, out))
            pr.start()
            running[pr] = (p, out, time.time())
        time.sleep(0.2)
        for pr, (p, out, t0) in list(running.items()):
            doc_id = os.path.basename(p)[:-4]
            if not pr.is_alive():
                pr.join()
                if os.path.exists(out):
                    try:
                        pages, err = json.load(gzip.open(out, "rt", encoding="utf-8"))
                        results[doc_id] = (pages, err)
                    except Exception as e:
                        results[doc_id] = (None, "bad result file: %s" % e)
                    os.remove(out)
                else:  # e.g. killed by the OOM killer
                    results[doc_id] = (None, "worker died (exit code %s)" % pr.exitcode)
                del running[pr]
            elif time.time() - t0 > per_doc_s:
                pr.kill()
                pr.join()
                results[doc_id] = (None, "timeout")
                if os.path.exists(out):
                    os.remove(out)
                del running[pr]
    return results


def verdict(pages):
    n = len(pages)
    if n == 0:
        return "empty"
    c = Counter(p["quality"] for p in pages)
    textual = n - c["empty"] - c["sparse"]
    if c["empty"] / n > 0.6:
        return "scanned_or_empty"
    if textual == 0:
        return "scanned_or_empty"
    if c["indic_unicode"] / textual > 0.5:
        return "indic_unicode"
    if (c["ok"] + c["numeric"]) / textual >= 0.7:
        return "good"
    if c["garbled"] / textual > 0.5:
        return "garbled_legacy_font"
    return "mixed"


# ----------------------------------------------------------------------------- driver
def load_status(out_dir):
    st = {}
    p = os.path.join(out_dir, "status.jsonl")
    if os.path.exists(p):
        for line in open(p, encoding="utf-8"):
            try:
                r = json.loads(line)
                st[r["doc_id"]] = r
            except Exception:
                pass
    return st


def run(manifest_path, out_dir, tmp_dir, workers=8, procs=4, batch=120, limit=0, max_hours=10.5,
        keep_pdfs_dir=None, retry_failed=False, only_ids=None):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(tmp_dir, exist_ok=True)
    docs = json.load(open(manifest_path, encoding="utf-8"))["docs"]
    if only_ids:
        docs = [d for d in docs if d["doc_id"] in only_ids]
    status = load_status(out_dir)
    todo = [d for d in docs if d["doc_id"] not in status
            or (retry_failed and status[d["doc_id"]]["status"] not in TERMINAL)]
    if limit:
        todo = todo[:limit]
    print("manifest %d | already done %d | to do %d" % (len(docs), len(status), len(todo)))
    shard = len([f for f in os.listdir(out_dir) if f.startswith("pages_")])
    t0 = time.time()
    sess = requests.Session()
    sess.mount("https://", requests.adapters.HTTPAdapter(pool_connections=workers, pool_maxsize=workers * 2))
    done = 0
    total_status, total_pages, total_bytes = Counter(), 0, 0
    log_path = os.path.join(out_dir, "progress.log")
    bar = _bar(len(todo))

    def log(msg):
        line = "%s | %s" % (time.strftime("%H:%M:%S"), msg)
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(line + "\n")
        return line

    def snapshot(stage):
        el = max(1e-9, time.time() - t0)
        rate = done / el
        eta = (len(todo) - done) / rate / 60 if rate > 0 else float("nan")
        snap = {"stage": stage, "done": done, "total": len(todo), "elapsed_min": round(el / 60, 1),
                "docs_per_min": round(rate * 60, 1), "eta_min": round(eta, 1), "status": dict(total_status),
                "pages": total_pages, "downloaded_mb": total_bytes // 2 ** 20,
                "disk_free_gb": round(shutil.disk_usage(out_dir).free / 2 ** 30, 1)}
        json.dump(snap, open(os.path.join(out_dir, "progress.json"), "w"))
        return snap

    log("start: %d docs to do, workers=%d, procs=%d, batch=%d" % (len(todo), workers, procs, batch))
    for b in range(0, len(todo), batch):
        if (time.time() - t0) / 3600 > max_hours:
            msg = log("time budget reached; stopping cleanly (rerun to resume)")
            print(msg)
            break
        chunk = todo[b:b + batch]
        bar.set_description("download")
        res = [None] * len(chunk)
        live = Counter()
        with ThreadPoolExecutor(workers) as ex:
            futs = {ex.submit(download, d, tmp_dir, sess): i for i, d in enumerate(chunk)}
            for fut in as_completed(futs):
                res[futs[fut]] = fut.result()
                live[fut.result()[0]] += 1
                bar.update(1)
                ok = total_status["ok"] + live["ok"]
                bad = sum(total_status.values()) - total_status["ok"] + sum(live.values()) - live["ok"]
                bar.set_postfix_str("ok=%d fail=%d last=%s" % (ok, bad, fut.result()[0]))
        bar.set_description("extract ")
        recs, paths = {}, []
        for d, (st, path, info) in zip(chunk, res):
            recs[d["doc_id"]] = {"doc_id": d["doc_id"], "status": st, "kind": d["kind"],
                                 "name": (d["names"] or [""])[0], "num_questions": d["num_questions"], **info}
            if path:
                paths.append(path)
        pages_out = []
        if paths:
            for doc_id, (pages, err) in extract_all(paths, procs).items():
                r = recs[doc_id]
                if err == "timeout":
                    r["status"], r["error"] = "extract_timeout", "killed after %ds" % EXTRACT_DOC_S
                elif err:
                    r["status"], r["error"] = "extract_error", err
                else:
                    r["n_pages"] = len(pages)
                    r["verdict"] = verdict(pages)
                    r["quality_counts"] = dict(Counter(p["quality"] for p in pages))
                    pages_out.extend(pages)
        for p in paths:
            if keep_pdfs_dir:
                os.makedirs(keep_pdfs_dir, exist_ok=True)
                shutil.move(p, os.path.join(keep_pdfs_dir, os.path.basename(p)))
            elif os.path.exists(p):
                os.remove(p)
        if pages_out:
            with gzip.open(os.path.join(out_dir, "pages_%05d.jsonl.gz" % shard), "wt", encoding="utf-8") as f:
                for p in pages_out:
                    f.write(json.dumps(p, ensure_ascii=False) + "\n")
            shard += 1
        with open(os.path.join(out_dir, "status.jsonl"), "a", encoding="utf-8") as f:
            for r in recs.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        done += len(chunk)
        c = Counter(r["status"] for r in recs.values())
        total_status.update(c)
        total_pages += len(pages_out)
        total_bytes += sum(r.get("bytes", 0) for r in recs.values())
        snap = snapshot("batch done")
        failed = [(r["doc_id"][:10], r["status"], r["num_questions"]) for r in recs.values() if r["status"] != "ok"]
        msg = log("batch %d/%d | %s | pages +%d | %.1f docs/min | ETA %.0f min | %d MB | disk free %.1f GB%s" % (
            done, len(todo), dict(c), len(pages_out), snap["docs_per_min"], snap["eta_min"],
            snap["downloaded_mb"], snap["disk_free_gb"],
            " | failed(id,status,nq): %s" % failed[:5] if failed else ""))
        tqdm_write(msg)
    bar.close()
    snapshot("finished")
    return summarize(out_dir)


def _bar(total):
    try:
        from tqdm.auto import tqdm
        return tqdm(total=total, unit="doc", dynamic_ncols=True)
    except Exception:
        class _B:  # minimal fallback without tqdm
            def update(self, n=1): pass
            def set_description(self, s): pass
            def set_postfix_str(self, s): pass
            def close(self): pass
        return _B()


def tqdm_write(msg):
    try:
        from tqdm.auto import tqdm
        tqdm.write(msg)
    except Exception:
        print(msg)


def summarize(out_dir):
    st = load_status(out_dir)
    path = os.path.join(out_dir, "doc_summary.csv")
    cols = ["doc_id", "status", "verdict", "n_pages", "num_questions", "bytes", "name", "filename", "kind"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        for r in st.values():
            w.writerow(r)
    print("status:", dict(Counter(r["status"] for r in st.values())))
    print("verdict:", dict(Counter(r.get("verdict", "-") for r in st.values() if r["status"] == "ok")))
    return st


Overwriting pdf_pipeline.py


In [11]:
import glob, os, shutil, json, importlib
import pdf_pipeline as pp
importlib.reload(pp)

cands = glob.glob("/kaggle/input/**/sources_manifest.json", recursive=True)
assert cands, "Attach the dataset that contains sources_manifest.json"
MANIFEST = cands[0]
OUT_DIR = "/kaggle/working/pdf_text"
TMP_DIR = "/tmp/pdfs"

# resume from a previous run's output if it was attached as an input
prev = glob.glob("/kaggle/input/**/pdf_text/status.jsonl", recursive=True)
if prev and not os.path.exists(os.path.join(OUT_DIR, "status.jsonl")):
    shutil.copytree(os.path.dirname(prev[0]), OUT_DIR, dirs_exist_ok=True)
    print("resumed from", os.path.dirname(prev[0]))
print("manifest:", MANIFEST, "| docs:", len(json.load(open(MANIFEST))["docs"]))

manifest: /kaggle/input/datasets/arth99/nlp-qna-manifest/sources_manifest.json | docs: 7428


## Smoke test: 6 docs, separate output dir (checks internet + endpoint + extraction)

In [12]:
SMOKE = "/kaggle/working/_smoke"
shutil.rmtree(SMOKE, ignore_errors=True)
pp.run(MANIFEST, SMOKE, TMP_DIR, workers=6, procs=2, batch=6, limit=6)
import gzip
for line in gzip.open(sorted(glob.glob(SMOKE + "/pages_*.jsonl.gz"))[0], "rt"):
    p = json.loads(line)
    if p["quality"] in ("ok", "indic_unicode"):
        print(p["doc_id"], "page", p["page"], p["script"], p["quality"], "\n", p["text"][:300]); break

manifest 7428 | already done 0 | to do 6


  0%|          | 0/6 [00:00<?, ?doc/s]

16:01:46 | batch 6/6 | {'ok': 6} | pages +1317 | 1.9 docs/min | ETA 0 min | 91 MB | disk free 19.5 GB
status: {'ok': 6}
verdict: {'scanned_or_empty': 1, 'garbled_legacy_font': 5}
zu51z4ee271ba7a4f4d05a7e266091f2608aa page 171 latin ok 
 161
ANNEXURE - I
Minimum Support Price of Different Crops  
	
	
	
	
	
	
	
	
          (Rs. per quintal)
Crop
Crop Year
2020-21
2021-22
2022-23
2023-24
2024-25
Paddy (Common)
1868
1940
2040
2183
2300
Paddy (Grade ‘A’)
1888
1960
2060
2203
2320
Jowar (Hybrid)
2620
2738
2970
3180
3371
Bajra
2150
2250
23


## Full run
`max_hours` stops cleanly before Kaggle's 12h session limit; re-run (with this version's output attached as input) to continue.
If many docs fail with `http_429`/timeouts, lower `workers`.

In [14]:
shutil.rmtree(SMOKE, ignore_errors=True)
st = pp.run(MANIFEST, OUT_DIR, TMP_DIR, workers=8, procs=4, batch=120, max_hours=10.0, retry_failed=True)

manifest 7428 | already done 1920 | to do 5513


  0%|          | 0/5513 [00:00<?, ?doc/s]

16:39:07 | batch 120/5513 | {'http_400': 6, 'ok': 114} | pages +3598 | 75.8 docs/min | ETA 71 min | 332 MB | disk free 19.4 GB | failed(id,status,nq): [('a5rv63a800', 'http_400', 13), ('4xs0r4582b', 'http_400', 6), ('7be2w8fe36', 'http_400', 6), ('h7tyn6ac8f', 'http_400', 6), ('pw6rdeebe1', 'http_400', 4)]
16:40:32 | batch 240/5513 | {'ok': 119, 'not_pdf': 1} | pages +4988 | 80.1 docs/min | ETA 66 min | 733 MB | disk free 19.4 GB | failed(id,status,nq): [('095h583be0', 'not_pdf', 2)]
16:42:01 | batch 360/5513 | {'ok': 116, 'not_pdf': 4} | pages +4751 | 80.1 docs/min | ETA 64 min | 1243 MB | disk free 19.4 GB | failed(id,status,nq): [('1aqxre9cee', 'not_pdf', 2), ('1obh96a3f3', 'not_pdf', 2), ('225vea8e2e', 'not_pdf', 2), ('40j8ace163', 'not_pdf', 2)]
16:43:16 | batch 480/5513 | {'ok': 120} | pages +3269 | 83.6 docs/min | ETA 60 min | 1544 MB | disk free 19.4 GB
16:44:53 | batch 600/5513 | {'ok': 108, 'not_pdf': 12} | pages +4894 | 81.5 docs/min | ETA 60 min | 2117 MB | disk free 19.4 G

In [15]:
!cd /kaggle/working && zip -0 -r pdf_text.zip pdf_text && ls -lh pdf_text.zip && wc -l pdf_text/status.jsonl

  adding: pdf_text/ (stored 0%)
  adding: pdf_text/pages_00011.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00036.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00010.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00017.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00042.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00055.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00012.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00033.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00026.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00046.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00059.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00015.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00029.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00018.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00047.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00004.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00019.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00003.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00032.j

In [6]:

import csv, glob, os, json, gzip, random, collections as C
OUT_DIR = "/kaggle/working/pdf_text"
st = {}
for line in open(OUT_DIR + "/status.jsonl", encoding="utf-8"):
    try: r = json.loads(line); st[r["doc_id"]] = r
    except Exception: pass
cols = ["doc_id","status","verdict","n_pages","num_questions","bytes","name","filename","kind"]
with open(OUT_DIR + "/doc_summary.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore"); w.writeheader(); w.writerows(st.values())
print("docs with status:", len(st), "of 7428")

tot = sum(int(r.get("num_questions") or 0) for r in st.values()) or 1
by = C.defaultdict(lambda: [0, 0])
for r in st.values():
    k = r.get("verdict") if r["status"] == "ok" else r["status"]
    by[k][0] += 1; by[k][1] += int(r.get("num_questions") or 0)
print("%-24s %6s %14s" % ("outcome", "docs", "citation share"))
for k, (n, q) in sorted(by.items(), key=lambda kv: -kv[1][1]):
    print("%-24s %6d %13.1f%%" % (k, n, 100 * q / tot))

garbled = [d for d, r in st.items() if r.get("verdict") == "garbled_legacy_font"]
random.seed(0); sample = set(random.sample(garbled, 8)); shown = C.Counter()
for f in sorted(glob.glob(OUT_DIR + "/pages_*.jsonl.gz")):
    for line in gzip.open(f, "rt", encoding="utf-8"):
        p = json.loads(line)
        if p["doc_id"] in sample and p["quality"] == "garbled" and shown[p["doc_id"]] < 1:
            shown[p["doc_id"]] += 1; r = st[p["doc_id"]]
            print("=" * 80)
            print((r.get("filename") or r.get("name") or "")[:90], "| pages", r["n_pages"], "| nq", r["num_questions"], "| sw", p["sw"], "| script", p["script"])
            print(repr(p["text"][:350]))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/pdf_text/status.jsonl'

## Report: how much of the citation mass is usable?

In [17]:
import csv, collections as C
rows = list(csv.DictReader(open(OUT_DIR + "/doc_summary.csv", encoding="utf-8")))
tot = sum(int(r["num_questions"] or 0) for r in rows) or 1
by = C.defaultdict(lambda: [0, 0])
for r in rows:
    k = r["verdict"] if r["status"] == "ok" else r["status"]
    by[k][0] += 1
    by[k][1] += int(r["num_questions"] or 0)
print("%-24s %6s %14s" % ("outcome", "docs", "citation share"))
for k, (n, q) in sorted(by.items(), key=lambda kv: -kv[1][1]):
    print("%-24s %6d %13.1f%%" % (k, n, 100 * q / tot))
print("\noutput size (MB):", sum(os.path.getsize(f) for f in glob.glob(OUT_DIR + "/*")) // 2**20)

outcome                    docs citation share
garbled_legacy_font        5173          75.8%
scanned_or_empty           1861          21.0%
not_pdf                     197           1.2%
indic_unicode                65           1.1%
http_400                    100           0.5%
mixed                        24           0.3%
good                          4           0.0%
extract_timeout               1           0.0%
http_404                      1           0.0%
timeout                       1           0.0%
empty                         1           0.0%

output size (MB): 159


In [4]:
import csv, glob, os, json, gzip, collections as C
OUT_DIR = "/kaggle/working/pdf_text"

# rebuild doc_summary.csv straight from status.jsonl (last record per doc wins)
st = {}
for line in open(OUT_DIR + "/status.jsonl", encoding="utf-8"):
    try:
        r = json.loads(line); st[r["doc_id"]] = r
    except Exception:
        pass
cols = ["doc_id", "status", "verdict", "n_pages", "num_questions", "bytes", "name", "filename", "kind"]
with open(OUT_DIR + "/doc_summary.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore"); w.writeheader()
    for r in st.values(): w.writerow(r)
print("docs with a status record:", len(st), "of 7428")

tot = sum(int(r.get("num_questions") or 0) for r in st.values()) or 1
by = C.defaultdict(lambda: [0, 0])
for r in st.values():
    k = r.get("verdict") if r["status"] == "ok" else r["status"]
    by[k][0] += 1
    by[k][1] += int(r.get("num_questions") or 0)
print("%-24s %6s %14s" % ("outcome", "docs", "citation share"))
for k, (n, q) in sorted(by.items(), key=lambda kv: -kv[1][1]):
    print("%-24s %6d %13.1f%%" % (k, n, 100 * q / tot))

seen, dup = set(), 0
for f in sorted(glob.glob(OUT_DIR + "/pages_*.jsonl.gz")):
    for line in gzip.open(f, "rt", encoding="utf-8"):
        p = json.loads(line); key = (p["doc_id"], p["page"])
        dup += key in seen; seen.add(key)
print("\npages:", len(seen), "| duplicate pages:", dup)
print("output size (MB):", sum(os.path.getsize(f) for f in glob.glob(OUT_DIR + "/*")) // 2**20)
print("extract_timeout docs:", [(r["doc_id"][:10], r.get("name", "")[:40], r["num_questions"]) for r in st.values() if r["status"] == "extract_timeout"])

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/pdf_text/status.jsonl'

## After it finishes
*Save Version -> Save & Run All (Commit)*, then on the output tab choose **New Dataset** to publish `pdf_text/`
(shards `pages_*.jsonl.gz`, `status.jsonl`, `doc_summary.csv`). Phase 1 step 6-9 (chunking, translation, indexing) read from that dataset.

In [5]:
!wc -l /kaggle/working/pdf_text/status.jsonl
!ls /kaggle/working/pdf_text | tail -3

wc: /kaggle/working/pdf_text/status.jsonl: No such file or directory
ls: cannot access '/kaggle/working/pdf_text': No such file or directory


In [13]:
!cd /kaggle/working && zip -0 -r pdf_text_backup.zip pdf_text && ls -lh pdf_text_backup.zip

  adding: pdf_text/ (stored 0%)
  adding: pdf_text/pages_00015.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00009.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00021.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00006.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00012.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00003.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00023.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00026.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00007.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00022.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00010.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00002.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00018.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00016.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00011.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00008.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00004.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00013.jsonl.gz (stored 0%)
  adding: pdf_text/pages_00025.j

In [16]:
 !pkill -9 -f multiprocessing

In [16]:
import os, json, shutil, subprocess
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = s.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = s.get_secret("KAGGLE_KEY")
d = "/kaggle/working/ds"; os.makedirs(d, exist_ok=True)
shutil.copytree("/kaggle/working/pdf_text", d + "/pdf_text", dirs_exist_ok=True)
json.dump({"title": "nlp-qna-pdf-text", "id": "arth99/nlp-qna-pdf-text",
           "licenses": [{"name": "CC0-1.0"}]}, open(d + "/dataset-metadata.json", "w"))
print(subprocess.run(["kaggle", "datasets", "create", "-p", d, "--dir-mode", "zip"], capture_output=True, text=True))

CompletedProcess(args=['kaggle', 'datasets', 'create', '-p', '/kaggle/working/ds', '--dir-mode', 'zip'], returncode=0, stdout="Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.2), please consider upgrading to the latest version (2.2.5)\nStarting upload for file pdf_text.zip\nUpload successful: pdf_text.zip (156MB)\nYour private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/arth99/nlp-qna-pdf-text\n", stderr='\n  0%|          | 0.00/156M [00:00<?, ?B/s]\n  6%|▋         | 9.95M/156M [00:00<00:01, 104MB/s]\n 18%|█▊        | 28.3M/156M [00:00<00:00, 147MB/s]\n 27%|██▋       | 42.2M/156M [00:00<00:00, 131MB/s]\n 35%|███▌      | 54.8M/156M [00:00<00:00, 131MB/s]\n 43%|████▎     | 67.3M/156M [00:00<00:00, 126MB/s]\n 51%|█████     | 79.3M/156M [00:00<00:00, 114MB/s]\n 61%|██████▏   | 96.0M/156M [00:00<00:00, 131MB/s]\n 70%|██████▉   | 109M/156M [00:00<00:00, 125MB/s] \n 78%|███████▊  | 122M/156M [00:01<00:00, 128MB/s]\n 86%|████